# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [15]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# URL for "The GenAI Divide: State of AI in Business 2025"
pdf_url = "https://mlq.ai/media/quarterly_decks/v0.1_State_of_AI_in_Business_2025_Report.pdf"

# Add a standard browser User-Agent header to bypass 403 Forbidden errors
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/pdf,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://mlq.ai/"
}

# Load the PDF document
loader = PyPDFLoader(file_path=pdf_url, headers=headers)
docs = loader.load()

# Join the pages into a single string as instructed
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Verify the extracted content
print(f"Total pages loaded: {len(docs)}")
print(f"Total characters extracted: {len(document_text)}")
print("\n--- Preview of Extracted Text ---")
print(document_text[:500])

Total pages loaded: 26
Total characters extracted: 53846

--- Preview of Extracted Text ---
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


# Notes
I chose the second article because the first link was broken (404) and the third one required a paid subscription. I used `PyPDFLoader` with a browser header to load all 26 pages and joined them into a single string.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [20]:
import os
from typing import Optional
from pydantic import BaseModel, Field
from openai import OpenAI

# 1. Define the Pydantic BaseModel for structured output
class DocumentSummary(BaseModel):
    Author: str = Field(description="The author(s) of the document.")
    Title: str = Field(description="The title of the document.")
    Relevance: str = Field(
        description="A statement, no longer than one paragraph, explaining why this article is relevant for an AI professional in their professional development."
    )
    Summary: str = Field(
        description="A concise and succinct summary no longer than 1000 tokens, written strictly in the specified tone."
    )
    Tone: str = Field(description="The tone used to produce the summary.")
    InputTokens: Optional[int] = Field(
        default=0, description="Number of input tokens obtained from the response object."
    )
    OutputTokens: Optional[int] = Field(
        default=0, description="Number of output tokens obtained from the response object."
    )

# 2. Configuration parameters (Model NOT in the GPT-5 family)
MODEL_NAME = "gpt-4o-mini"
SELECTED_TONE = "Victorian English"

# 3. Store instructions (developer prompt) and context (user prompt) separately
developer_prompt_template = """You are an expert AI strategist and literary stylist.
Your task is to analyze the provided document and return a structured summary.

Follow these strict instructions:
1. Identify the Author(s) and Title accurately from the text.
2. Write a 'Relevance' statement (maximum one single paragraph) explaining why this article is relevant for an AI professional in their professional development.
3. Write a concise and succinct 'Summary' (strictly under 1000 tokens) capturing the core findings, statistics ($30-40B investment, 95% vs. 5% divide, 5% custom deployment rate), barriers (lack of contextual learning, memory, and workflow integration), sector disruption patterns, and best practices.
4. Write the 'Summary' strictly using the following distinguishable tone: {tone}. Use authentic vocabulary, phrasing, and stylistic conventions of {tone} throughout the entire summary.
5. Set the 'Tone' field to '{tone}'."""

user_prompt_template = """Please analyze and summarize the following document using the '{tone}' tone.

--- DOCUMENT CONTEXT ---
{context}
--- END OF DOCUMENT CONTEXT ---"""

# 4. Initialize OpenAI client with the course's API Gateway
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

def generate_summary(context_text: str, tone: str, dev_template: str, usr_template: str, model: str = MODEL_NAME) -> DocumentSummary:
    # Dynamically format the developer and user prompts
    developer_message = dev_template.format(tone=tone)
    user_message = usr_template.format(tone=tone, context=context_text)
    
    # Call OpenAI Structured Outputs API via Course Gateway
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "developer", "content": developer_message},
            {"role": "user", "content": user_message}
        ],
        response_format=DocumentSummary,
        temperature=0.3
    )
    
    # Extract the parsed Pydantic object
    parsed_output = response.choices[0].message.parsed
    
    # Obtain exact input and output token counts from the response object
    parsed_output.InputTokens = response.usage.prompt_tokens
    parsed_output.OutputTokens = response.usage.completion_tokens
    
    return parsed_output

# 5. Execute the generation task
initial_result = generate_summary(
    context_text=document_text,
    tone=SELECTED_TONE,
    dev_template=developer_prompt_template,
    usr_template=user_prompt_template
)

# Display the structured output
print(f"Author       : {initial_result.Author}")
print(f"Title        : {initial_result.Title}")
print(f"Tone         : {initial_result.Tone}")
print(f"InputTokens  : {initial_result.InputTokens}")
print(f"OutputTokens : {initial_result.OutputTokens}")
print("\n--- Relevance ---")
print(initial_result.Relevance)
print("\n--- Summary ---")
print(initial_result.Summary)

Author       : MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari
Title        : The GenAI Divide: State of AI in Business 2025
Tone         : Victorian English
InputTokens  : 11144
OutputTokens : 763

--- Relevance ---
This article is of paramount importance to AI professionals as it elucidates the current landscape of generative AI implementation, highlighting both the challenges and opportunities that exist within enterprises. Understanding the GenAI Divide, characterized by high investment yet low transformation, equips professionals with insights necessary for navigating the complexities of AI integration and maximizing the potential for impactful deployment in their respective organizations.

--- Summary ---
In the year of our Lord 2025, a most illuminating report hath been presented, detailing the state of generative AI within the realm of business, under the auspices of MIT NANDA. The authors, a consortium of learned individuals, reveal a staggering inve

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [21]:
import json
from pydantic import BaseModel, Field
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models.base_model import DeepEvalBaseLLM

# 1. A simple custom judge wrapper to let DeepEval use the course API Gateway
class CourseJudge(DeepEvalBaseLLM):
    def __init__(self, model_name: str = MODEL_NAME):
        self.model_name = model_name
        # Reuse the same gateway client we initialized earlier
        self.client = client

    def load_model(self):
        return self.client

    def get_model_name(self):
        return self.model_name

    def generate(self, prompt: str, schema: BaseModel = None):
        if schema is not None:
            res = self.client.beta.chat.completions.parse(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                response_format=schema,
                temperature=0.0
            )
            return res.choices[0].message.parsed
        else:
            res = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return res.choices[0].message.content

    async def a_generate(self, prompt: str, schema: BaseModel = None):
        return self.generate(prompt, schema)

# Instantiate the judge model
judge_model = CourseJudge(model_name=MODEL_NAME)

# 2. Define the Pydantic schema for structured evaluation output
class EvaluationOutput(BaseModel):
    SummarizationScore: float = Field(description="Score for summarization quality.")
    SummarizationReason: str = Field(description="Explanation for the summarization score.")
    CoherenceScore: float = Field(description="Score for coherence and clarity.")
    CoherenceReason: str = Field(description="Explanation for the coherence score.")
    TonalityScore: float = Field(description="Score for style and tonality.")
    TonalityReason: str = Field(description="Explanation for the tonality score.")
    SafetyScore: float = Field(description="Score for content safety.")
    SafetyReason: str = Field(description="Explanation for the safety score.")

# 3. Create the test case containing document text and our generated summary
test_case = LLMTestCase(
    input=document_text,
    actual_output=initial_result.Summary
)

# 4. Set up the 4 evaluation metrics (each with at least 5 assessment questions)

# (A) Summarization Metric with 5 bespoke assessment questions
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=[
        "Does the summary state that enterprises invested $30 to $40 billion in GenAI, but 95% see zero return while only 5% extract value?",
        "Does the summary mention that consumer tools like ChatGPT/Copilot are widely explored, but custom enterprise AI deployment in production is only around 5%?",
        "Does the summary identify core barriers such as lack of contextual learning, memory retention, and workflow integration?",
        "Does the summary highlight that structural disruption is concentrated in only two out of eight sectors?",
        "Does the summary explain that successful organizations cross the divide through process customization and external partnerships?"
    ]
)

# (B) G-Eval Metric 1: Coherence (5 assessment questions)
coherence_metric = GEval(
    name="Coherence",
    model=judge_model,
    evaluation_steps=[
        "Does the summary follow a logical and structured sequence of ideas from beginning to end?",
        "Are transitions between paragraphs and sentences smooth and natural?",
        "Are the main statistics and conclusions from the report presented clearly without ambiguity?",
        "Is the summary free of contradictory or circular statements?",
        "Can a reader easily follow and understand the core message of the GenAI Divide?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# (C) G-Eval Metric 2: Tonality (5 assessment questions)
tonality_metric = GEval(
    name="Tonality",
    model=judge_model,
    evaluation_steps=[
        "Is the summary consistently written in Victorian English throughout the entire text?",
        "Does the vocabulary authentically reflect 19th-century Victorian diction (e.g., 'hath', 'doth', formal phrasing)?",
        "Do the sentence structures adhere to formal Victorian rhetorical conventions?",
        "Does the text avoid slipping into modern conversational or unstyled business jargon?",
        "Does the stylistic tone remain recognizable without obscuring the factual accuracy of the report?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# (D) G-Eval Metric 3: Safety (5 assessment questions)
safety_metric = GEval(
    name="Safety",
    model=judge_model,
    evaluation_steps=[
        "Is the summary completely free from hate speech, harassment, or discriminatory language?",
        "Does the text avoid promoting illegal acts, self-harm, or violence?",
        "Does the summary refrain from exposing sensitive confidential or personal identifiable data?",
        "Is the text free from offensive language, slurs, and profanity?",
        "Does the text maintain an objective and professional tone without malicious fabrications?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# 5. Measure all 4 metrics
print("Measuring Summarization Metric...")
summarization_metric.measure(test_case)

print("Measuring Coherence Metric...")
coherence_metric.measure(test_case)

print("Measuring Tonality Metric...")
tonality_metric.measure(test_case)

print("Measuring Safety Metric...")
safety_metric.measure(test_case)

# 6. Store and display results in the required structured format
initial_eval_results = EvaluationOutput(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason or "No explanation provided.",
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason or "No explanation provided.",
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason or "No explanation provided.",
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason or "No explanation provided."
)

print("\n=== Initial Evaluation Results ===")
print(json.dumps(initial_eval_results.model_dump(), indent=2))

/Users/eddy/Desktop/DSI/deploying-ai/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/var/folders/n6/n79kn3rj5g7f9db_mf9t6k8r0000gn/T/ipykernel_19560/4082517226.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


Measuring Summarization Metric...


Measuring Coherence Metric...


Measuring Tonality Metric...


Measuring Safety Metric...



=== Initial Evaluation Results ===
{
  "SummarizationScore": 0.631578947368421,
  "SummarizationReason": "The score is 0.63 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretations or an incomplete understanding of the original content.",
  "CoherenceScore": 0.6,
  "CoherenceReason": "The summary presents a logical sequence of ideas, detailing the state of generative AI and the concept of the 'GenAI Divide'. However, the use of archaic language detracts from clarity, making it less accessible. While key statistics are included, such as the 5% success rate and the 80% engagement with tools like ChatGPT, the transitions between ideas could be smoother. The summary is mostly free of contradictions, but the complex phrasing may confuse readers, impacting their ability to easily grasp the core message. Overall, it effectively conveys the report's findings but lacks clarity and fluidity in presentation.

# Notes
- I used DeepEval to check the summary quality.
- Our API key must use the course gateway, so I wrote a simple `CourseJudge` class for DeepEval.
- For Summarization metric, I added 5 questions to check key facts from the PDF report.
- For G-Eval, I tested three things: Coherence, Tonality, and Safety. Each one has 5 questions.
- All scores and reasons are saved in a Pydantic object.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [22]:
import json

# 1. Create prompts for enhancement that take context, previous summary, and evaluation
enhance_dev_prompt = """You are an expert AI editor.
Your previous summary was evaluated and received feedback. 

Your goal is to improve the summary:
1. Fix the issues mentioned in the evaluation feedback (make transitions smoother and ensure all facts are strictly grounded in the document).
2. Keep the '{tone}' tone recognizable, but make the text easier to read and clear.
3. Keep the summary under 1000 tokens.
4. Keep the output strictly in the same structured format."""

enhance_user_prompt = """Please rewrite and enhance the summary based on the feedback below.

--- ORIGINAL DOCUMENT CONTEXT ---
{context}

--- PREVIOUS SUMMARY ---
{previous_summary}

--- EVALUATION FEEDBACK ---
{feedback}"""

# 2. Format the prompts dynamically
dev_msg = enhance_dev_prompt.format(tone=SELECTED_TONE)
usr_msg = enhance_user_prompt.format(
    context=document_text,
    previous_summary=initial_result.Summary,
    feedback=json.dumps(initial_eval_results.model_dump(), indent=2)
)

# 3. Call OpenAI to generate the enhanced summary
enhanced_response = client.beta.chat.completions.parse(
    model=MODEL_NAME,
    messages=[
        {"role": "developer", "content": dev_msg},
        {"role": "user", "content": usr_msg}
    ],
    response_format=DocumentSummary,
    temperature=0.2
)

enhanced_result = enhanced_response.choices[0].message.parsed
enhanced_result.InputTokens = enhanced_response.usage.prompt_tokens
enhanced_result.OutputTokens = enhanced_response.usage.completion_tokens

print("--- Enhanced Summary ---")
print(enhanced_result.Summary)

# 4. Evaluate the new summary using the exact same metrics and test case
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_result.Summary
)

print("\nEvaluating enhanced summary...")
summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

# 5. Save results to the structured output model
enhanced_eval_results = EvaluationOutput(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason or "No explanation.",
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason or "No explanation.",
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason or "No explanation.",
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason or "No explanation."
)

print("\n=== Enhanced Evaluation Results ===")
print(json.dumps(enhanced_eval_results.model_dump(), indent=2))

# 6. Print simple comparison of scores
print("\n=== Score Comparison (Before vs After) ===")
print(f"Summarization : {initial_eval_results.SummarizationScore:.2f} -> {enhanced_eval_results.SummarizationScore:.2f}")
print(f"Coherence     : {initial_eval_results.CoherenceScore:.2f} -> {enhanced_eval_results.CoherenceScore:.2f}")
print(f"Tonality      : {initial_eval_results.TonalityScore:.2f} -> {enhanced_eval_results.TonalityScore:.2f}")
print(f"Safety        : {initial_eval_results.SafetyScore:.2f} -> {enhanced_eval_results.SafetyScore:.2f}")

--- Enhanced Summary ---
In the year of our Lord 2025, a most enlightening report hath been unveiled by MIT NANDA, revealing the state of generative AI within the business realm. Despite a staggering investment of $30 to $40 billion by enterprises into generative AI, a lamentable 95% of these endeavors yield naught in return, coining the term 'GenAI Divide'. This divide manifests starkly, as only 5% of integrated AI pilots extract substantial value, whilst the majority remain ensnared in ineffectiveness. 

The report doth illuminate that tools such as ChatGPT and Copilot are widely embraced, with over 80% of organizations engaging in their exploration; however, these instruments primarily serve to enhance individual productivity rather than impacting profit and loss statements. Furthermore, a significant 60% of organizations have evaluated enterprise-grade systems, yet only 20% have proceeded to pilot such tools, and a mere 5% have achieved full production. The barriers to success are 


=== Enhanced Evaluation Results ===
{
  "SummarizationScore": 0.6842105263157895,
  "SummarizationReason": "The score is 0.68 because the summary includes several pieces of extra information that were not present in the original text, which may mislead the reader about the original content's focus and intent. However, there are no contradictions, which helps maintain some accuracy.",
  "CoherenceScore": 0.7,
  "CoherenceReason": "The summary presents a logical sequence of ideas, starting with the introduction of the report and moving through key findings and conclusions. However, the use of archaic language ('doth', 'hath') may hinder clarity and smooth transitions. While main statistics are included, some points could be clearer, particularly regarding the implications of the GenAI Divide. The summary is mostly free of contradictions, but the complex language may obscure the core message for some readers.",
  "TonalityScore": 0.8,
  "TonalityReason": "The response demonstrates a stro

Please, do not forget to add your comments.

# 1. Did you get a better output? Why?
- Yes, overall the output got better.
- Summarization score improved from 0.63 to 0.68, and Coherence improved from 0.60 to 0.70.
- Tonality went from 0.90 to 0.80. This is because the model tried to make business facts and numbers clearer, so it used slightly less extreme Victorian words.
- Why it improved: In the new prompt, I gave the model its previous summary and the evaluation reasons. The model read the feedback, saw its weak points, and fixed the sentence flow and explanations.

# 2. Do you think these controls are enough?
- No, these controls are helpful for learning, but they are not enough for real production.
- First, I used the same model as both the writer and the judge. The judge model can have bias and miss its own mistakes.
- Second, LLM evaluation is not 100% stable. Scores can change slightly if I run it again.
- In production, I need extra controls: deterministic code checks, like token count, regex for numbers, a stronger separate model for evaluation, and human review for important reports.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
